## Colab 준비
Colab에서 열었다면 **아래 칸을 맨 먼저 실행**하세요. 도우미 파일과 한글 글꼴을 받아요 (10~20초).

- 더 빠르게: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**
- 고친 내용 저장: **파일 → Drive에 사본 저장**
- 맥 실습실에서 열었다면 이 칸은 아무 일도 안 해요.

In [ ]:
import os, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/sangwon9401/silicon-descent/main/colab/"
    for f in ['common.py', 'solutions/__init__.py', 'solutions/sol00.py', 'solutions/sol01.py', 'solutions/sol02.py', 'solutions/sol04.py']:
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + f, f)
    os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")
    print("Colab 준비 완료 ✅")
else:
    print("Colab이 아니에요. 그냥 넘어가요.")

# 03. 내 맥에서 Roofline 그려 보기

**목표**: "메모리 병목"과 "연산 병목"을 내 컴퓨터에서 **숫자로 직접** 확인해요.

사이트의 강의 "성능의 첫 원리"를 먼저 보고 오세요. 이번 실습은 코드 채우기가 적고, 주로 **실행하고 관찰**해요.

⚠️ 측정 중에는 다른 무거운 프로그램을 잠시 쉬게 하면 결과가 더 깔끔해요.

In [ ]:
import matplotlib.pyplot as plt
import torch
from common import *

setup_korean_font()
device = pick_device()
dtype = torch.float16
print("장치:", device)

## 1. 메모리 대역폭 재기: 벡터 덧셈

`c = a + b`는 원소 하나에 덧셈 1번이라, 거의 **메모리를 옮기는 속도**만 재게 돼요.
옮긴 바이트 = 읽기 2개 + 쓰기 1개 = 원소당 3 × 2바이트(FP16).

In [ ]:
for n in [1 << 20, 1 << 24, 1 << 26]:
    a = torch.randn(n, device=device, dtype=dtype); b = torch.randn_like(a); c = torch.empty_like(a)
    t = timeit(lambda: torch.add(a, b, out=c), device, repeat=20)
    print(f"원소 {n:>11,}개: {3 * n * 2 / t / 1e9:7.1f} GB/s")
bw = 3 * n * 2 / t / 1e9
print(f"\n→ 내 맥의 메모리 대역폭 측정값: 약 {bw:.0f} GB/s")

## 2. 최대 연산 성능 재기: 큰 행렬곱

n × n 행렬 두 개를 곱하면 계산 횟수는 `2 × n³`이에요. 행렬이 클수록 연산 강도가 높아서 연산기를 꽉 채울 수 있어요.

In [ ]:
for n in [512, 1024, 2048, 4096]:
    A = torch.randn(n, n, device=device, dtype=dtype); B = torch.randn_like(A)
    t = timeit(lambda: A @ B, device, repeat=10)
    print(f"{n:>5} × {n:<5}: {2 * n**3 / t / 1e12:6.2f} TFLOPS")
peak = 2 * n**3 / t / 1e12
print(f"\n→ 내 맥의 최대 연산 성능 측정값: 약 {peak:.1f} TFLOPS (FP16)")

## 3. 내 맥의 꺾임점 계산하기

꺾임점 = 최대 연산 성능 ÷ 메모리 대역폭 (단위를 맞추면 FLOP/바이트).
직접 계산식을 채워 보세요. `peak`은 TFLOPS(×10¹²), `bw`는 GB/s(×10⁹)예요.

In [ ]:
ridge = ...  # 여기를 채우세요

# 막히면 아래 줄 맨 앞의 '#'을 지우세요.
# ridge = peak * 1e12 / (bw * 1e9)

In [ ]:
run_check("꺾임점", lambda: check_close("꺾임점", ridge, peak * 1e12 / (bw * 1e9), rtol=1e-3))
print(f"내 맥은 바이트당 약 {peak * 1e12 / (bw * 1e9):.0f}번 계산해야 연산기가 쉬지 않아요.")

## 4. 디코드 흉내 내기: 배치를 키우면?

디코드는 `[배치, 4096]` 입력에 `[4096, 4096]` 가중치를 곱하는 것과 같아요.
- 가중치를 한 번 읽는 비용(32MB)은 배치와 상관없이 같아요.
- 계산량은 배치에 비례해요.

그래서 배치가 작을 땐 **시간이 거의 그대로**(메모리 병목)이고, 어느 순간부터 **시간이 배치에 비례해서** 늘어나요(연산 병목).

먼저 연산 강도 공식을 채워 보세요. 계산 = `2 × b × n × n`, 옮긴 바이트 ≈ 가중치 `n × n × 2` + 입력/출력 `2 × b × n × 2`.

In [ ]:
def intensity_matvec(b, n):
    ...  # 여기를 채우세요 (계산 ÷ 바이트)


# 막히면:
# def intensity_matvec(b, n): return (2 * b * n * n) / (n * n * 2 + 2 * b * n * 2)

In [ ]:
run_check("연산 강도", lambda: check_close("연산 강도", intensity_matvec(8, 4096),
                                        2 * 8 * 4096**2 / (4096**2 * 2 + 2 * 8 * 4096 * 2), rtol=1e-4))

In [ ]:
n = 4096
Wt = torch.randn(n, n, device=device, dtype=dtype)
batches = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
times, tflops = [], []
for b in batches:
    x = torch.randn(b, n, device=device, dtype=dtype)
    t = timeit(lambda: x @ Wt.T, device, warmup=5, repeat=50)
    times.append(t * 1e6); tflops.append(2 * b * n * n / t / 1e12)
    print(f"배치 {b:>4}: {t*1e6:8.1f} µs, {tflops[-1]:6.2f} TFLOPS, 연산 강도 {2*b*n*n/(n*n*2+2*b*n*2):6.1f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(batches, times, "o-"); ax[0].set_xscale("log", base=2)
ax[0].set_xlabel("배치"); ax[0].set_ylabel("한 번 계산 시간 (µs)"); ax[0].set_title("배치가 커져도 처음엔 시간이 거의 그대로")
ax[1].plot(batches, tflops, "o-", color="C1"); ax[1].axhline(peak, ls="--", color="gray", label=f"최대 {peak:.1f} TFLOPS")
ax[1].set_xscale("log", base=2); ax[1].set_xlabel("배치"); ax[1].set_ylabel("TFLOPS"); ax[1].set_title("처리 성능은 꺾임점까지 올라가요"); ax[1].legend()
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 정리

- 왼쪽 그래프에서 시간이 거의 평평한 구간이 **메모리 병목** 구간이에요. 배치 1이든 16이든 가중치를 읽는 시간이 대부분이라서요.
- 오른쪽 그래프에서 성능이 점선(최대치) 근처로 붙는 지점이 꺾임점이에요.
- 맥은 통합 메모리라서 NVIDIA GPU와 숫자는 다르지만, **모양은 똑같아요**. 사이트의 '추론 계산기' 탭에서 GPU를 바꿔 보며 비교해 보세요.

**생각해 보기**: 서버가 요청을 많이 모아서 한 번에 처리(배칭)하면 왜 전체 처리량이 오르는지, 이 그래프로 설명해 보세요.

다음: **04_sampling.ipynb**